In [104]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings("ignore")
import pickle
import os

In [105]:
liverFunction = pd.read_csv(r"C:\Users\direk\Disease_risk_predictor_-3\SYSTEM\dataset\liver_function.csv")
liverFunction.head()

,alt,ast,alp,bilirubin_total,bilirubin_direct,albumin,total_protein,liver_disease_risk,fatty_liver_risk,hepatitis_risk
0,26.78,26.68,50.19,0.88,0.28,4.53,7.44,0.00,21.04,6.98
1,49.53,38.00,126.05,0.49,0.07,4.84,7.59,22.92,62.22,0.00
2,23.98,12.12,94.61,1.51,0.34,4.54,8.03,18.93,15.42,20.79
3,21.83,17.27,67.76,1.12,0.29,4.12,6.78,7.81,7.75,0.00
4,22.58,10.75,118.63,1.11,0.22,4.13,6.34,1.51,10.18,0.00


In [106]:
def liver_disease(row):
    score = 0

    if row["alt"] > 150 or row["ast"] > 150:
        return "Critical"

    if row["alt"] > 80 or row["ast"] > 80:
        score += 2
    elif row["alt"] > 40 or row["ast"] > 40:
        score += 1
    
    if row["bilirubin_total"] > 2:
        score += 1

    if row["albumin"] < 3:
        score += 1

    if score >= 3:
        return "High"
    elif score >= 1:
        return "Medium"
    else:
        return "Low"


def fatty_liver(row):
    if row["alt"] > 120:
        return "Critical"
    elif row["alt"] > row["ast"] and row["alt"] > 50:
        return "High"
    elif row["alt"] > row["ast"]:
        return "Medium"
    else:
        return "Low"


def hepatitis(row):
    if row["alt"] > 200 or row["ast"] > 200:
        return "Critical"
    elif row["alt"] > 100 or row["ast"] > 100:
        return "High"
    elif row["alt"] > 60 or row["ast"] > 60:
        return "Medium"
    else:
        return "Low"

In [107]:
liverFunction["liver_disease"] = liverFunction.apply(liver_disease, axis=1)
liverFunction["fatty_liver"] = liverFunction.apply(fatty_liver, axis=1)
liverFunction["hepatitis"] = liverFunction.apply(hepatitis, axis=1)
liverFunction.head()

,alt,ast,alp,bilirubin_total,bilirubin_direct,albumin,total_protein,liver_disease_risk,fatty_liver_risk,hepatitis_risk,liver_disease,fatty_liver,hepatitis
0,26.78,26.68,50.19,0.88,0.28,4.53,7.44,0.00,21.04,6.98,Low,Medium,Low
1,49.53,38.00,126.05,0.49,0.07,4.84,7.59,22.92,62.22,0.00,Medium,Medium,Low
2,23.98,12.12,94.61,1.51,0.34,4.54,8.03,18.93,15.42,20.79,Low,Medium,Low
3,21.83,17.27,67.76,1.12,0.29,4.12,6.78,7.81,7.75,0.00,Low,Medium,Low
4,22.58,10.75,118.63,1.11,0.22,4.13,6.34,1.51,10.18,0.00,Low,Medium,Low


In [108]:
LABEL_MAPPING = {
    "Low": "low",
    "Normal": "moderate",
    "Medium": "moderate",
    "High": "high",
    "Critical": "critical"
}

NUM_MAPPING = {
    "low": 0,
    "moderate": 1,
    "high": 2,
    "critical": 3
}

In [109]:
liverFunction["liver_disease"] = liverFunction["liver_disease"].map(LABEL_MAPPING)
liverFunction["fatty_liver"] = liverFunction["fatty_liver"].map(LABEL_MAPPING)
liverFunction["hepatitis"] = liverFunction["hepatitis"].map(LABEL_MAPPING)
liverFunction.head()

,alt,ast,alp,bilirubin_total,bilirubin_direct,albumin,total_protein,liver_disease_risk,fatty_liver_risk,hepatitis_risk,liver_disease,fatty_liver,hepatitis
0,26.78,26.68,50.19,0.88,0.28,4.53,7.44,0.00,21.04,6.98,low,moderate,low
1,49.53,38.00,126.05,0.49,0.07,4.84,7.59,22.92,62.22,0.00,moderate,moderate,low
2,23.98,12.12,94.61,1.51,0.34,4.54,8.03,18.93,15.42,20.79,low,moderate,low
3,21.83,17.27,67.76,1.12,0.29,4.12,6.78,7.81,7.75,0.00,low,moderate,low
4,22.58,10.75,118.63,1.11,0.22,4.13,6.34,1.51,10.18,0.00,low,moderate,low


In [110]:
liverFunction["liver_disease_num"] = liverFunction["liver_disease"].map(NUM_MAPPING)
liverFunction["fatty_liver_num"] = liverFunction["fatty_liver"].map(NUM_MAPPING)
liverFunction["hepatitis_num"] = liverFunction["hepatitis"].map(NUM_MAPPING)
liverFunction.head()

,alt,ast,alp,bilirubin_total,bilirubin_direct,albumin,total_protein,liver_disease_risk,fatty_liver_risk,hepatitis_risk,liver_disease,fatty_liver,hepatitis,liver_disease_num,fatty_liver_num,hepatitis_num
0,26.78,26.68,50.19,0.88,0.28,4.53,7.44,0.00,21.04,6.98,low,moderate,low,0,1,0
1,49.53,38.00,126.05,0.49,0.07,4.84,7.59,22.92,62.22,0.00,moderate,moderate,low,1,1,0
2,23.98,12.12,94.61,1.51,0.34,4.54,8.03,18.93,15.42,20.79,low,moderate,low,0,1,0
3,21.83,17.27,67.76,1.12,0.29,4.12,6.78,7.81,7.75,0.00,low,moderate,low,0,1,0
4,22.58,10.75,118.63,1.11,0.22,4.13,6.34,1.51,10.18,0.00,low,moderate,low,0,1,0


In [111]:
"""Preparing model of Ml prediction"""
feature_cols = ["alt", "ast", "alp", "bilirubin_total", "bilirubin_direct", "albumin","total_protein"]
X = liverFunction[feature_cols]
y_liver_disease = liverFunction["liver_disease_num"]
y_fatty_liver = liverFunction["fatty_liver_num"]
y_hepatitis = liverFunction["hepatitis_num"]

In [112]:
"""train test split""" 
X_train, X_test, y_train_liver_disease, y_test_liver_disease = train_test_split(X, y_liver_disease, test_size=0.2, random_state=31, stratify=y_liver_disease)
_, _, y_train_fatty_liver, y_test_fatty_liver = train_test_split(X, y_fatty_liver,  test_size=0.2, random_state=31, stratify=y_fatty_liver) 
_, _, y_train_hepatitis, y_test_hepatitis = train_test_split(X, y_hepatitis,  test_size=0.2, random_state=31, stratify=y_hepatitis)

In [113]:
"""Auto detect classes and print report - works for any number of classes"""
CLASS_NAMES = {0: "low", 1: "moderate", 2: "high", 3: "critical"}

def print_report(y_test, y_pred, model_name):
    # automatically finds which classes exist in test + predictions
    existing_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
    existing_names = [CLASS_NAMES[i] for i in existing_labels]

    print("=" * 40)
    print(f"{model_name} MODEL ACCURACY")
    print("=" * 40)
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print()
    print("=" * 40)
    print(f"{model_name} CLASSIFICATION REPORT")
    print("=" * 40)
    print(classification_report(
        y_test, y_pred,
        labels=existing_labels,
        target_names=existing_names
    ))

In [114]:
"""Choosing the best hyperparameters for """ 
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3]
}

model = XGBClassifier(
    random_state=31,
    use_label_encoder=False,
    eval_metric="mlogloss"
)

In [115]:
"""fatty liver modal""" 
grid_fatty_liver = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_fatty_liver.fit(X_train, y_train_fatty_liver)
print("Best Params:", grid_fatty_liver.best_params_)
print("Best Score:", grid_fatty_liver.best_score_)
print()
print()
model_fatty_liver = grid_fatty_liver.best_estimator_
y_pred_fatty_liver = model_fatty_liver.predict(X_test)
print_report(y_test_fatty_liver, y_pred_fatty_liver, "FATTY LIVER DISEASE")

Best Params: {'colsample_bytree': 0.7, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}
Best Score: 0.875


FATTY LIVER DISEASE MODEL ACCURACY
Accuracy: 88.00%

FATTY LIVER DISEASE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.00      0.00      0.00         8
    moderate       0.88      1.00      0.94        88
        high       0.00      0.00      0.00         4

    accuracy                           0.88       100
   macro avg       0.29      0.33      0.31       100
weighted avg       0.77      0.88      0.82       100



In [116]:
"""Liver disease modal"""
grid_liver_disease = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_liver_disease.fit(X_train, y_train_liver_disease)
print("Best Params:", grid_liver_disease.best_params_)
print("Best Score:", grid_liver_disease.best_score_)
print()
print()
model_liver_disease = grid_liver_disease.best_estimator_
y_pred_liver_disease = model_liver_disease.predict(X_test)
print_report(y_test_liver_disease, y_pred_liver_disease, "LIVER DISEASE")

Best Params: {'colsample_bytree': 0.7, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.7}
Best Score: 0.9974999999999999


LIVER DISEASE MODEL ACCURACY
Accuracy: 100.00%

LIVER DISEASE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       1.00      1.00      1.00        80
    moderate       1.00      1.00      1.00        20

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [117]:
"""Hepatitis modal"""
grid_hepatitis = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_hepatitis.fit(X_train, y_train_hepatitis)
print("Best Params:", grid_hepatitis.best_params_)
print("Best Score:", grid_hepatitis.best_score_)
print()
print()
model_hepatitis = grid_hepatitis.best_estimator_
y_pred_hepatitis = model_hepatitis.predict(X_test)
print_report(y_test_hepatitis, y_pred_hepatitis, "HEPATITIS DISEASE")

Best Params: {'colsample_bytree': 0.7, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}
Best Score: 0.9924999999999999


HEPATITIS DISEASE MODEL ACCURACY
Accuracy: 99.00%

HEPATITIS DISEASE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.99      1.00      0.99        99
    moderate       0.00      0.00      0.00         1

    accuracy                           0.99       100
   macro avg       0.49      0.50      0.50       100
weighted avg       0.98      0.99      0.99       100



In [119]:
"""saving the modals""" 
save_path = r"C:\Users\direk\Disease_risk_predictor_-3\ml_models\xgboost"
os.makedirs(save_path, exist_ok=True)
with open(os.path.join(save_path, "liver.pkl"), "wb") as f:
    pickle.dump(model_liver_disease, f)
    print("liver.pkl is saved successfully")
with open(os.path.join(save_path, "fatty_liver.pkl"), "wb") as f:
    pickle.dump(model_fatty_liver, f)
    print("fatty_liver.pkl is saved successfully")
with open(os.path.join(save_path, "hepatitis.pkl"), "wb") as f:
    pickle.dump(model_hepatitis, f)
    print("hepatitis.pkl is saved successfully")

liver.pkl is saved successfully
fatty_liver.pkl is saved successfully
hepatitis.pkl is saved successfully
